
# 🧑‍🎓 Projeto — Clusterização a partir dos dados brutos de Distribuição de Renda

Faculdade de Tecnologia e Inovação Senac DF  
Curso: **Tecnologia em Ciência de Dados**  
Autores: **Anderson de Matos Guimarães, Gustavo Stefano Thomazinho e Renan Ost**  
Professor orientador: **Rogério Gomes Lopes**

Este notebook **replica a lógica e a estrutura do notebook do professor (projeto IDEB)**, adaptando-a ao dataset bruto de **Distribuição de Renda**.


**Paradigma do professor (3 variáveis):**
1. **Evolução relativa** *(ex.: Var_IDEB = IDEB_final / IDEB_inicial)*  
2. **Nível inicial** *(ex.: IDEB_2005)*  
3. **Nível atual** *(ex.: IDEB_2023)*

**Adaptação para Renda (RTB):**
1. **Var_RTB = RTB_final / RTB_inicial**
2. **RTB_inicial = rtb_soma_centil no ano inicial**
3. **RTB_final = rtb_soma_centil no ano final**

O notebook:
- Lê o **CSV bruto**,
- Seleciona **dois anos** (preferência: 2010 e 2020; se indisponíveis, usa o **mínimo** e o **máximo** dos anos existentes),
- Cria `Var_RTB`, `rtb_inicial` e `rtb_final`,
- Padroniza as 3 variáveis e realiza **Método do Cotovelo + K-Means**,
- Gera **tabela-resumo** e **gráficos no estilo do professor** (padrão com `matplotlib`).

> Observação: Mantemos nomenclaturas como `colunasSelecionadas`, `colunasSelecionadasCluster` e `colunasSelecionadasPadronizadas` para ficar **o mais próximo possível** do notebook original.


## 1) Imports, constantes e caminhos

In [ ]:

from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Reprodutibilidade
RANDOM_SEED = 42 # Qualquer inteiro serve, porém use sempre o mesmo para garantir reprodutibilidade
np.random.seed(RANDOM_SEED) 

# Caminhos - informam onde estão os dados
RAW_CSV = Path("data/raw/distribuicao-renda.csv")  # Caminho padrão de projeto - onde os dados "deveriam" estar
RAW_CSV_FALLBACK = Path("/mnt/data/distribuicao-renda.csv")  # fallback: arquivo anexado nesta sessão - onde os dados "podem" estar (válido para o Google Colab) 
OUT_DIR = Path("data/processed") # Onde salvar os dados processados (criados por este notebook)
OUT_DIR.mkdir(parents=True, exist_ok=True) # Garante que o diretório de saída exista

# Resolução do caminho de entrada
if RAW_CSV.exists():
    CSV_IN = RAW_CSV
elif RAW_CSV_FALLBACK.exists():
    CSV_IN = RAW_CSV_FALLBACK
else:
    raise FileNotFoundError("Arquivo CSV bruto não encontrado. Ajuste RAW_CSV ou coloque o arquivo em /mnt/data/distribuicao-renda.csv")

print(f"[OK] Lendo CSV bruto: {CSV_IN}")


## 2) Leitura dos dados brutos e padronização de colunas

In [ ]:

df_raw = pd.read_csv(CSV_IN) 

# Colunas esperadas mínimas: ano, uf, centil, rtb_soma_centil
colunasEsperadasMin = {"ano","uf","centil","rtb_soma_centil"}
faltam = colunasEsperadasMin - set(map(str.lower, df_raw.columns))
if faltam:
    print("[ATENÇÃO] Colunas mínimas ausentes (verifique nomes/snake_case):", faltam)

# Forçar snake_case simples
df_raw.columns = (
    df_raw.columns.str.strip()
                  .str.lower()
                  .str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True)
                  .str.replace("__+", "_", regex=True)
                  .str.strip("_")
)

display(df_raw.head())
print("shape:", df_raw.shape)
print("anos únicos:", sorted(df_raw["ano"].unique()))
print("UFs (amostra):", df_raw["uf"].dropna().unique()[:10])


## 3) Definição de anos (inicial e final) e criação de `rtb_inicial`, `rtb_final`, `Var_RTB`

In [ ]:

# Preferência: usar 2010 e 2020; se indisponíveis, cair para min/max
anos_disponiveis = sorted(df_raw["ano"].dropna().astype(int).unique().tolist())
preferidos = (2010, 2020)

if all(a in anos_disponiveis for a in preferidos):
    ANO_INICIAL, ANO_FINAL = preferidos
else:
    ANO_INICIAL, ANO_FINAL = anos_disponiveis[0], anos_disponiveis[-1]

print(f"[INFO] ANO_INICIAL={ANO_INICIAL} | ANO_FINAL={ANO_FINAL} (ajuste conforme necessário)")

# Selecionar colunas-base
colunasSelecionadas = [
    "ano",
    "uf",
    "centil",
    "rtb_soma_centil",
    # mantemos bens/dividas apenas como contexto (não entram nos 3 clusters principais)
    *([c for c in ["bens_imoveis","dividas_onus"] if c in df_raw.columns])
]

dados = df_raw[colunasSelecionadas].dropna(subset=["ano","uf","centil","rtb_soma_centil"]).copy()
dados["ano"] = dados["ano"].astype(int)

# Pivot para ter colunas lado a lado por (uf, centil)
pivot = dados.pivot_table(
    index=["uf","centil"],
    columns="ano",
    values="rtb_soma_centil",
    aggfunc="sum"
)

# Garantir existência das colunas do ano inicial/final
anos_cols = pivot.columns.tolist()
if ANO_INICIAL not in anos_cols or ANO_FINAL not in anos_cols:
    raise ValueError(f"Anos {ANO_INICIAL} e/ou {ANO_FINAL} não disponíveis nas colunas pivotadas: {anos_cols[:10]} ...")

# Construir dataframe final de trabalho
df_work = pivot[[ANO_INICIAL, ANO_FINAL]].rename(
    columns={ANO_INICIAL:"rtb_inicial", ANO_FINAL:"rtb_final"}
).reset_index()

# Evitar divisão por zero: se rtb_inicial==0, definimos Var_RTB = np.nan (ou 0); aqui optamos por np.nan e depois imputamos 0
df_work["Var_RTB"] = df_work.apply(
    lambda r: (r["rtb_final"]/r["rtb_inicial"]) if r["rtb_inicial"] not in [0, None, np.nan] else np.nan,
    axis=1
)

# Imputação simples (como o professor costuma aplicar fillna em algumas etapas)
df_work[["rtb_inicial","rtb_final","Var_RTB"]] = df_work[["rtb_inicial","rtb_final","Var_RTB"]].fillna(0)

display(df_work.head())
print("shape:", df_work.shape)


## 4) Enriquecimento (Região Geográfica) e padronização (StandardScaler)

In [ ]:

# Mapeamento UF -> Região Geográfica (simplificado)
REGIOES = {
    "AC":"Norte","AP":"Norte","AM":"Norte","PA":"Norte","RO":"Norte","RR":"Norte","TO":"Norte",
    "AL":"Nordeste","BA":"Nordeste","CE":"Nordeste","MA":"Nordeste","PB":"Nordeste",
    "PE":"Nordeste","PI":"Nordeste","RN":"Nordeste","SE":"Nordeste",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MT":"Centro-Oeste","MS":"Centro-Oeste",
    "ES":"Sudeste","MG":"Sudeste","RJ":"Sudeste","SP":"Sudeste",
    "PR":"Sul","RS":"Sul","SC":"Sul"
}
df_work["Regiao_Geografica"] = df_work["uf"].map(REGIOES).fillna("Ignorado")

# Seleção das 3 variáveis de clusterização (paradigma IDEB)
colunasSelecionadasCluster = ["Var_RTB","rtb_inicial","rtb_final"]

# Padronização (sufixo _PADRONIZADA)
scaler = StandardScaler()
X_std = scaler.fit_transform(df_work[colunasSelecionadasCluster])

dadosPadronizada = pd.concat(
    [df_work.reset_index(drop=True),
     pd.DataFrame(X_std, columns=[c + "_PADRONIZADA" for c in colunasSelecionadasCluster])],
    axis=1
)

colunasSelecionadasPadronizadas = [c + "_PADRONIZADA" for c in colunasSelecionadasCluster]

display(dadosPadronizada.head())


## 5) Método do Cotovelo (WCSS) — escolha de *k*

In [ ]:

wcss = []
maxCluster = 10  # mesmo espírito do professor

for i in range(1, maxCluster):
    kmeans = KMeans(
        n_clusters=i, random_state=RANDOM_SEED, init="k-means++",
        n_init=10, max_iter=1000, tol=1e-8
    )
    kmeans.fit(dadosPadronizada[colunasSelecionadasPadronizadas])
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, maxCluster), wcss, marker="o")
plt.title("Método do Cotovelo (WCSS)")
plt.xlabel("Número de Clusters (k)")
plt.ylabel("WCSS")
plt.grid(True)
plt.show()


## 6) K-Means final e rótulos (A, B, C, ...)

In [ ]:

# Defina k conforme o cotovelo. Padrão didático: 4 (como no exemplo do professor)
k = 4

kmeans = KMeans(
    n_clusters=k, random_state=RANDOM_SEED, init="k-means++",
    n_init=10, max_iter=10000, tol=1e-8
)
kmeans.fit(dadosPadronizada[colunasSelecionadasPadronizadas])

dadosPadronizada["cluster_num"] = kmeans.predict(dadosPadronizada[colunasSelecionadasPadronizadas])
dadosPadronizada["cluster"] = dadosPadronizada["cluster_num"].apply(lambda x: chr(x + 65))

# Tabela-resumo (medianas por cluster + contagem)
df_cluster_info = dadosPadronizada.groupby("cluster")[colunasSelecionadasCluster].median().reset_index()
df_cluster_count = dadosPadronizada.groupby("cluster").size().reset_index(name="count")
df_cluster_table = pd.merge(df_cluster_info, df_cluster_count, on="cluster", how="left").sort_values("cluster")

display(df_cluster_table)


## 7) Rótulos semânticos (opcional) — interpretação

In [ ]:

# Heurística simples para nomear clusters por mediana de rtb_final (só como exemplo didático)
order = df_cluster_table.sort_values("rtb_final").cluster.tolist()
labels = {cl: f"Perfil {i+1} (rtb_final rank {i+1})" for i, cl in enumerate(order)}
dadosPadronizada["DS_cluster"] = dadosPadronizada["cluster"].map(labels)

display(dadosPadronizada[["uf","centil","rtb_inicial","rtb_final","Var_RTB","cluster","DS_cluster"]].head(10))


## 8) Gráficos — distribuição por cluster e composições

In [ ]:

# 8.1 Histogramas (aproximação do KDE do professor, usando matplotlib)
vars_plot = ["Var_RTB","rtb_inicial","rtb_final"]

for var in vars_plot:
    plt.figure(figsize=(8,5))
    for cl in sorted(dadosPadronizada["cluster"].unique()):
        subset = dadosPadronizada[dadosPadronizada["cluster"]==cl][var]
        plt.hist(subset, bins=20, alpha=0.5, label=f"Cluster {cl}", density=True)
    plt.title(f"Distribuição — {var} por cluster")
    plt.xlabel(var)
    plt.ylabel("Densidade")
    plt.legend()
    plt.grid(True)
    plt.show()

# 8.2 Pizza por Região dentro de cada cluster (função estilo 'plot_pie_charts')
def plot_pie_charts(df, category_col, split_col):
    clusters = sorted(df[split_col].unique())
    for cl in clusters:
        tmp = df[df[split_col]==cl][category_col].value_counts().sort_values(ascending=False)
        plt.figure(figsize=(5,5))
        plt.pie(tmp.values, labels=tmp.index, autopct="%1.1f%%", startangle=90)
        plt.title(f"Composição de {category_col} — Cluster {cl}")
        plt.tight_layout()
        plt.show()

plot_pie_charts(dadosPadronizada, category_col="Regiao_Geografica", split_col="cluster")

# 8.3 Gráfico radar (função estilo 'make_spider')
def make_spider(stats: pd.Series, title: str):
    categories = list(stats.index)
    N = len(categories)

    # Fechar o polígono
    values = stats.values.tolist()
    values += values[:1]

    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig = plt.figure(figsize=(6,6))
    ax = plt.subplot(111, polar=True)
    ax.plot(angles, values)
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_yticklabels([])
    plt.title(title)
    plt.show()

# Radar por cluster com as 3 variáveis padronizadas (medianas)
radar_stats = dadosPadronizada.groupby("cluster")[colunasSelecionadasPadronizadas].median()
for cl, row in radar_stats.iterrows():
    make_spider(row, f"Radar — Cluster {cl}")


## 9) Salvar saídas

In [ ]:

OUT_CSV = OUT_DIR / "renda_clusterizada_ideb_like.csv"
dadosPadronizada.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"[SALVO] {OUT_CSV}")

# Metadados do experimento
import json
meta = {
    "anos": {"inicial": int(ANO_INICIAL), "final": int(ANO_FINAL)},
    "features": colunasSelecionadasCluster,
    "features_padronizadas": colunasSelecionadasPadronizadas,
    "random_seed": RANDOM_SEED,
    "k": int(k),
    "input_csv": str(CSV_IN)
}
with open(OUT_DIR / "renda_cluster_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(f"[SALVO] {OUT_DIR / 'renda_cluster_meta.json'}")
